<a href="https://colab.research.google.com/github/Reybolouri/Semester-Project-ECON8310-Reybolouri/blob/main/forcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas statsmodels openpyxl


In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202022%20Student%20Sign%20In%20and%20Out.xlsx"
df2022 = pd.read_excel(url, engine="openpyxl", header=[0,1], skiprows=5)
# …etc.



In [5]:

!pip install --quiet pandas numpy statsmodels openpyxl



In [6]:

import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1) Loader + cleaner for one ECEC file
def load_and_clean_ecec_file(file_path, year):
    # Read with two header rows and skip first 5 metadata lines
    df_raw = pd.read_excel(
        file_path, header=[0,1], skiprows=5, engine="openpyxl"
    )
    # Split metadata vs. timestamp columns
    meta_df       = df_raw.iloc[:, :7].copy()
    timestamps_df = df_raw.iloc[:, 7:].copy()
    # Flatten column names
    meta_df.columns = [
        '_'.join([str(c) for c in col if 'Unnamed' not in str(c)]).strip()
        for col in meta_df.columns
    ]
    timestamps_df.columns = [
        f"{str(c[0]).strip()}_{str(c[1]).strip()}"
        for c in timestamps_df.columns
    ]
    # Add row_id for merging
    meta_df['row_id']       = meta_df.index
    timestamps_df['row_id'] = timestamps_df.index
    # Melt wide → long
    long_df = timestamps_df.melt(
        id_vars='row_id', var_name='date_inout', value_name='timestamp'
    )
    long_df[['date_str','in_out']] = long_df['date_inout']\
        .str.extract(r'(.*)_(IN|OUT)', expand=True)
    # Parse date (month/day + year)
    long_df['date_str'] = long_df['date_str'].str.strip() + f" {year}"
    long_df['date']     = pd.to_datetime(
        long_df['date_str'], format='%b %d %Y', errors='coerce'
    )
    # Pivot so that each row_id+date has one IN and one OUT
    pivot_df = long_df.pivot_table(
        index=['row_id','date'], columns='in_out',
        values='timestamp', aggfunc='first'
    ).reset_index()
    # Ensure both columns exist
    for c in ['IN','OUT']:
        if c not in pivot_df.columns:
            pivot_df[c] = pd.NaT
    # Merge back metadata
    final_df = pivot_df.merge(meta_df, on='row_id', how='left')
    # Clean raw time strings
    def clean_time_only(x):
        if pd.isna(x): return x
        m = re.match(r'^\s*\d{1,2}:\d{2}\s*(AM|PM)', str(x), re.IGNORECASE)
        return m.group(0) if m else x
    final_df['IN']  = final_df['IN'].apply(clean_time_only)
    final_df['OUT'] = final_df['OUT'].apply(clean_time_only)
    final_df['year'] = int(year)
    # Return core columns
    return final_df[['Record ID','Student Status','Room','Tags','date','IN','OUT','year']]

# 2) Function to expand sessions into 30-min blocks and compute staffing
def build_staffing_grid(all_years_df):
    # Extract age_group
    pattern = r'(Infants|Multi-Age|Toddlers|Preschool|Pre-K)'
    all_years_df['age_group'] = all_years_df['Room'].str.extract(pattern, expand=False)
    # Treat “--” as missing OUT
    all_years_df['OUT'] = all_years_df['OUT'].replace('--', pd.NA)
    # Build full datetime fields
    all_years_df['in_datetime'] = pd.to_datetime(
        all_years_df['date'].dt.strftime('%Y-%m-%d') + ' ' + all_years_df['IN'],
        format='%Y-%m-%d %I:%M %p', errors='coerce'
    )
    all_years_df['out_datetime'] = pd.to_datetime(
        all_years_df['date'].dt.strftime('%Y-%m-%d') + ' ' + all_years_df['OUT'],
        format='%Y-%m-%d %I:%M %p', errors='coerce'
    )
    # Drop incomplete sessions
    attended = all_years_df.dropna(subset=['in_datetime','out_datetime']).copy()
    # Generate 30-min blocks
    def blocks(s,e):
        return pd.date_range(start=s, end=e, freq='30min').tolist()
    attended['time_blocks'] = attended.apply(
        lambda r: blocks(r['in_datetime'],r['out_datetime']), axis=1
    )
    expl = attended.explode('time_blocks')
    expl['time_block'] = expl['time_blocks'].dt.floor('30min')
    grid = expl[['Record ID','Student Status','age_group','time_block']].copy()
    # Count children & apply ratios
    grp = grid.groupby(['age_group','time_block','Student Status'])\
              .agg(children_present=('Record ID','nunique'))\
              .reset_index()
    ratios = pd.DataFrame({
        'age_group':['Infants','Multi-Age','Toddlers','Preschool','Pre-K'],
        'student_to_staff':[4,4,6,10,12]
    })
    grp = grp.merge(ratios,on='age_group',how='left')
    grp['staff_required'] = np.ceil(grp['children_present']/grp['student_to_staff']).astype(int)
    return grp



In [7]:

ecec_files = {
    "2022":"https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202022%20Student%20Sign%20In%20and%20Out.xlsx",
    "2023":"https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202023%20Student%20Sign%20In%20and%20Out.xlsx",
    "2024":"https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202024%20Student%20Sign%20In%20and%20Out.xlsx",
    "2025":"https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202025%2001012025-02282025%20Student%20Sign%20In%20and%20Out.xlsx"
}

# Load + concatenate
all_years = pd.concat(
    [load_and_clean_ecec_file(url, yr) for yr,url in ecec_files.items()],
    ignore_index=True
)
# Quick check
print("Rows after cleaning:", len(all_years))

# Build staffing grid
staffing = build_staffing_grid(all_years)
print("Rows in 30-min staffing grid:", staffing.shape[0])

# Save intermediate
staffing.to_csv("ecec_staffing_grouped.csv", index=False)


Rows after cleaning: 61472
Rows in 30-min staffing grid: 122413


In [8]:
#Typical Week & Next-Week Forecast + Accuracy
# Load the grid
df = pd.read_csv("ecec_staffing_grouped.csv", parse_dates=["time_block"])
# Build time series of total staff needed
ts = (
    df[df["Student Status"]=="Active"]
    .groupby("time_block")["staff_required"]
    .sum()
    .sort_index()
)

# Split train/test
h = 7*48
train = ts.iloc[:-h]
test  = ts.iloc[-h:]

# 1) SARIMAX next-week
model = SARIMAX(train,
                order=(1,0,1),
                seasonal_order=(1,1,1,336),
                enforce_stationarity=False,
                enforce_invertibility=False)
res = model.fit(disp=False)
fc_sarima = res.get_forecast(steps=h).predicted_mean

# 2) Typical-week average
df_ts = ts.to_frame("staff")
df_ts["dow"]=df_ts.index.dayofweek
df_ts["hr"]=df_ts.index.hour
df_ts["mn"]=df_ts.index.minute
typ = df_ts.groupby(["dow","hr","mn"])["staff"].mean()
fc_typ = test.index.to_series().apply(
    lambda dt: typ.loc[(dt.dayofweek, dt.hour, dt.minute)]
)

# Accuracy metrics
import numpy as np
def metrics(f, t):
    mae  = np.mean(np.abs(f-t))
    rmse = np.sqrt(np.mean((f-t)**2))
    mape = np.mean(np.abs((f-t)/t))*100
    return mae, rmse, mape

mae_s, rmse_s, mape_s = metrics(fc_sarima, test)
mae_t, rmse_t, mape_t = metrics(fc_typ,   test)

print("\nAccuracy on hold-out week:")
print(f"SARIMAX    → MAE={mae_s:.2f}, RMSE={rmse_s:.2f}, MAPE={mape_s:.1f}%")
print(f"Typical Wk → MAE={mae_t:.2f}, RMSE={rmse_t:.2f}, MAPE={mape_t:.1f}%")

# Save forecasts
pd.Series(fc_sarima, name="SARIMAX").to_csv("next_week_sarima.csv")
pd.Series(fc_typ,   name="Typical").to_csv("next_week_typical.csv")
pd.Series(ts[-336:], name="Actual").to_csv("actual_week.csv")


/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


KeyboardInterrupt: 